## Round Robin Scheduling Algorithm Demo

Demo for Round Robin scheduling algorithm with 8 processes


In [6]:
import sys, os
sys.path.append(os.path.abspath(".."))  # add parent directory to search path

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from core.process import Process
from core.scheduler_rr import round_robin
import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display


## Create Sample Processes

creating 8 processes with different burst times and arrival times to run the basic Round Robin scheduler.

In [8]:
# Create 8 processes with varying burst times and arrival times
processes = [
    Process(pid="P1", burst_time=6, arrival_time=0),
    Process(pid="P2", burst_time=4, arrival_time=1),
    Process(pid="P3", burst_time=8, arrival_time=2),
    Process(pid="P4", burst_time=3, arrival_time=3),
    Process(pid="P5", burst_time=7, arrival_time=4),
    Process(pid="P6", burst_time=5, arrival_time=5),
    Process(pid="P7", burst_time=4, arrival_time=6),
    Process(pid="P8", burst_time=6, arrival_time=7)
]

# Display process information
process_df = pd.DataFrame([
    {
        'Process ID': p.pid,
        'Burst Time': p.burst_time,
        'Arrival Time': p.arrival_time
    } for p in processes
])

display(process_df)

,Process ID,Burst Time,Arrival Time
0,P1,6,0
1,P2,4,1
2,P3,8,2
3,P4,3,3
4,P5,7,4
5,P6,5,5
6,P7,4,6
7,P8,6,7


## Run the Round Robin algorithm

Running the round robin algorithm with the processes and time quantum of 3

In [9]:
time_quantum = 3
context_switch_time = 1
metrics = round_robin(processes, time_quantum, context_switch_time)

# Create a DataFrame with completion metrics
results_df = pd.DataFrame([
    {
        'Process ID': p.pid,
        'Arrival Time': p.arrival_time,
        'Burst Time': p.burst_time,
        'Completion Time': p.completion_time,
        'Turnaround Time': p.turnaround_time,
        'Waiting Time': p.waiting_time,
        'Start Time': p.start_time,
        'First Response Time': metrics["first_response_times"].get(p.pid, None),
    } for p in metrics['completed_processes']
])

display(results_df)

rq_df = pd.DataFrame(metrics['rq_length_over_time'], columns=['Time', 'RQ Length'])

display(rq_df.head())

,Process ID,Arrival Time,Burst Time,Completion Time,Turnaround Time,Waiting Time,Start Time,First Response Time
0,P4,3,3,16,13,10,12,9
1,P1,0,6,24,24,18,0,0
2,P2,1,4,38,37,33,4,3
3,P6,5,5,49,44,39,24,19
4,P7,6,4,51,45,41,28,22
5,P8,7,6,55,48,42,32,25
6,P3,2,8,58,56,48,8,6
7,P5,4,7,59,55,48,16,12


,Time,RQ Length
0,0,1
1,4,4
2,4,5
3,8,7
4,8,8


## Metrics

Visualizing the waiting times and turnaround times for the processes.

In [10]:
# Create bar chart comparing waiting times and turnaround times
fig = go.Figure()

fig.add_trace(go.Bar(
    x=results_df['Process ID'],
    y=results_df['Waiting Time'],
    name='Waiting Time',
    marker_color='rgb(55, 83, 109)'
))

fig.add_trace(go.Bar(
    x=results_df['Process ID'],
    y=results_df['Turnaround Time'],
    name='Turnaround Time',
    marker_color='rgb(26, 118, 255)'
))

fig.update_layout(
    title='Process Waiting and Turnaround Times',
    xaxis_title='Process ID',
    yaxis_title='Time Units',
    barmode='group',
    bargap=0.15,
    bargroupgap=0.1
)

fig.show()

# Print average metrics
print(f"Average Turnaround Time: {metrics['average_turnaround_time']:.2f}")
print(f"Average Waiting Time: {metrics['average_waiting_time']:.2f}")
print(f"CPU Utilization: {metrics['cpu_utilization']:.2f}%")
print(f"Context Switches: {metrics['context_switches']}")
print(f"Total Time: {metrics['execution_time']}")
print(f"Average First Response Time: {metrics['average_first_response_time']:.2f}")


if 'First Response Time' in results_df.columns:
    fig3 = go.Figure()
    fig3.add_trace(go.Bar(
        x=results_df['Process ID'],
        y=results_df['First Response Time'],
        name='First Response Time',
        marker_color='rgb(255, 153, 51)'
    ))
    fig3.update_layout(
        title='Process First Response Time',
        xaxis_title='Process ID',
        yaxis_title='Time Units',
        barmode='group'
    )
    fig3.show()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=rq_df["Time"],
    y=rq_df["RQ Length"],
    mode='lines+markers',
    name='Ready Queue Length',
    line=dict(color='orange')
))
fig2.update_layout(
    title='Ready Queue Length Over Time',
    xaxis_title='Time Units',
    yaxis_title='Queue Length'
)
fig2.show()

Average Turnaround Time: 40.25
Average Waiting Time: 34.88
CPU Utilization: 100.00%
Context Switches: 15
Total Time: 59
Average First Response Time: 12.00


## CPU Utilization

visualizing the CPU utilization and context switch overhead.

In [11]:
# Create pie chart for CPU utilization
labels = ['CPU Utilization', 'Context Switch Overhead', 'Idle Time']
values = [
    metrics['cpu_utilization'],
    metrics['context_switch_overhead'],
    100 - metrics['cpu_utilization'] - metrics['context_switch_overhead']
]

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_layout(title='CPU Time Distribution')
fig.show()